In [1]:
import pandas as pd
import numpy as np
import glob
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [2]:
def group(label):
    l = str(label).strip().lower()

    if l == "benign":
        return "BENIGN"

    if "web attack" in l:
        return "WebAttack"

    if l in ("ftp-patator", "ssh-patator"):
        return "BruteForce"

    if l.startswith("dos"):
        return "DoS"

    if l == "ddos":
        return "DDoS"

    if l == "portscan":
        return "PortScan"

    if l == "bot":
        return "Bot"

    return "Other"

In [3]:
train_parts = []
test_parts = []

files = sorted(glob.glob("../data/cicids2017/*.parquet"))

print("Files found:", len(files))

if not files:
    raise FileNotFoundError("No CICIDS2017 Parquet files found.")

for f in files:

    print("\nProcessing:", os.path.basename(f))

    d = pd.read_parquet(f)

    # Clean column names
    d.columns = d.columns.str.strip()

    # Convert numeric columns to float32
    num = d.select_dtypes(include="number").columns
    d[num] = d[num].astype("float32")

    # Remove invalid rows and duplicates while preserving row order
    d = d.replace([np.inf, -np.inf], np.nan)
    d = d.dropna()
    d = d.drop_duplicates()

    # Apply the original label grouping
    d["Group"] = d["Label"].apply(group)

    # Remove unsupported categories
    d = d[d["Group"] != "Other"]

    # Remove original Label column
    d = d.drop(columns=["Label"])

    # Split this file independently
    cut = int(len(d) * 0.70)

    train_parts.append(d.iloc[:cut].copy())
    test_parts.append(d.iloc[cut:].copy())

    print("Training rows:", cut)
    print("Testing rows:", len(d) - cut)

print("\nAll files processed.")

Files found: 8

Processing: Benign-Monday-no-metadata.parquet
Training rows: 321123
Testing rows: 137625

Processing: Botnet-Friday-no-metadata.parquet
Training rows: 123217
Testing rows: 52808

Processing: Bruteforce-Tuesday-no-metadata.parquet
Training rows: 272783
Testing rows: 116907

Processing: DDoS-Friday-no-metadata.parquet
Training rows: 154884
Testing rows: 66380

Processing: DoS-Wednesday-no-metadata.parquet
Training rows: 409458
Testing rows: 175483

Processing: Infiltration-Thursday-no-metadata.parquet
Training rows: 145297
Testing rows: 62271

Processing: Portscan-Friday-no-metadata.parquet
Training rows: 83665
Testing rows: 35857

Processing: WebAttacks-Thursday-no-metadata.parquet
Training rows: 109055
Testing rows: 46739

All files processed.


In [4]:
train = pd.concat(train_parts, ignore_index=True)
test = pd.concat(test_parts, ignore_index=True)

print("Training shape:", train.shape)
print("Testing shape:", test.shape)

print("\nTraining label distribution:")
print(train["Group"].value_counts())

print("\nTesting label distribution:")
print(test["Group"].value_counts())

Training shape: (1619482, 78)
Testing shape: (694070, 78)

Training label distribution:
Group
BENIGN        1322136
DoS            184570
DDoS           101019
BruteForce       7850
WebAttack        2143
Bot               987
PortScan          777
Name: count, dtype: int64

Testing label distribution:
Group
BENIGN        654986
DDoS           26995
DoS             9160
BruteForce      1300
PortScan        1179
Bot              450
Name: count, dtype: int64


In [5]:
cols = [
    c for c in train.columns
    if c != "Group" and train[c].nunique() > 1
]

print("Number of features:", len(cols))
print("Feature columns:", cols)

X_train = train[cols]
y_train = train["Group"]

X_test = test[cols]
y_test = test["Group"]

Number of features: 69
Feature columns: ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down

In [6]:
rf_ordered = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

rf_ordered.fit(X_train, y_train)

print("Time-ordered evaluation model trained successfully.")

Time-ordered evaluation model trained successfully.


In [7]:
predictions = rf_ordered.predict(X_test)

print("TIME-ORDERED EVALUATION RESULTS\n")

print(
    classification_report(
        y_test,
        predictions,
        digits=4,
        zero_division=0
    )
)

TIME-ORDERED EVALUATION RESULTS

              precision    recall  f1-score   support

      BENIGN     0.9946    0.9867    0.9907    654986
         Bot     0.0283    0.4889    0.0534       450
  BruteForce     0.9465    0.9669    0.9566      1300
        DDoS     0.9976    0.9977    0.9977     26995
         DoS     0.9064    0.6647    0.7670      9160
    PortScan     0.9469    0.8168    0.8770      1179
   WebAttack     0.0000    0.0000    0.0000         0

    accuracy                         0.9823    694070
   macro avg     0.6886    0.7031    0.6632    694070
weighted avg     0.9928    0.9823    0.9871    694070



In [13]:
from sklearn.metrics import confusion_matrix

# Convert BENIGN vs ATTACK
y_binary = (y_test != "BENIGN").astype(int)
pred_binary = (predictions != "BENIGN").astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_binary,
    pred_binary,
    labels=[0, 1]
).ravel()

print("ORDERED SPLIT RESULTS")
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

print(
    "False Positive Rate:",
    round(fp / (fp + tn), 6)
)

ORDERED SPLIT RESULTS
TN: 646298
FP: 8688
FN: 3490
TP: 35594
False Positive Rate: 0.013264


In [9]:
shared = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Fwd Packets Length Total",
    "Bwd Packets Length Total",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Avg Packet Size"
]

print("Shared features defined:", len(shared))

Shared features defined: 8


In [10]:
import pandas as pd

cic = pd.read_parquet("../data/clean.parquet")

missing = [c for c in shared if c not in cic.columns]

print("Total CICIDS2017 rows:", len(cic))
print("Shared features found:", len(shared) - len(missing))
print("Missing features:", missing)

Total CICIDS2017 rows: 2198955
Shared features found: 8
Missing features: []


In [11]:
# Binary view: BENIGN vs ATTACK
y_test_binary = (y_test != "BENIGN").astype(int)
pred_binary = (pred != "BENIGN").astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test_binary,
    pred_binary,
    labels=[0, 1]
).ravel()

print("Ordered split FPR:", round(fp / (fp + tn), 6))
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

NameError: name 'pred' is not defined